# Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Load and Preprocess Data

In [ ]:
# 데이터 로드
parking_monthly = pd.read_csv('../../hmw/Data/한강공원 주차장 월별 이용 현황.csv', encoding='cp949')
parking_daily = pd.read_csv('../../hmw/Data/한강공원 주차장 일별 이용 현황.csv', encoding='cp949')
real_time_parking = pd.read_csv('../../Data/서울시 시영주차장 실시간 주차대수 정보.csv', encoding='cp949')
subway_data = pd.read_csv('../../Data/서울시 지하철 호선별 역별 시간대별 승하차 인원 정보.csv', encoding='cp949')

# 난지한강공원 관련 데이터 필터링 (가정: 데이터에 '난지' 포함)
# 실제 데이터 구조에 따라 조정 필요
parking_daily = parking_daily[parking_daily['주차장명'].str.contains('난지', na=False)]

# 결측치 처리
parking_daily = parking_daily.dropna()

# 데이터 타입 변환
parking_daily['일자'] = pd.to_datetime(parking_daily['일자'], errors='coerce')

# Exploratory Data Analysis (EDA)

In [ ]:
# 데이터 요약
print("Daily Parking Data Info:")
print(parking_daily.info())
print(parking_daily.head())

# 통계 요약
print(parking_daily.describe())

# 시각화: 일별 이용 현황
plt.figure(figsize=(12, 6))
sns.lineplot(data=parking_daily, x='일자', y='주차대수')
plt.title('Daily Parking Occupancy')
plt.xticks(rotation=45)
plt.show()

# 상관관계 분석
plt.figure(figsize=(8, 6))
sns.heatmap(parking_daily.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# Feature Engineering

In [ ]:
# 실시간 데이터 사용 (시간별 가정)
# 데이터에 시간 컬럼이 있다고 가정
real_time_parking['datetime'] = pd.to_datetime(real_time_parking['기준시간'], errors='coerce')
real_time_parking = real_time_parking.dropna(subset=['datetime'])

# 난지 관련 필터링
real_time_parking = real_time_parking[real_time_parking['주차장명'].str.contains('난지', na=False)]

# 피처 생성
real_time_parking['hour'] = real_time_parking['datetime'].dt.hour
real_time_parking['day_of_week'] = real_time_parking['datetime'].dt.dayofweek
real_time_parking['month'] = real_time_parking['datetime'].dt.month

# 라그 피처: 1시간 전 주차대수
real_time_parking = real_time_parking.sort_values('datetime')
real_time_parking['lag_1h_parking'] = real_time_parking['주차대수'].shift(1)

# 총 주차 공간 수 가정 (데이터에 없으면 추정)
total_spaces = real_time_parking['주차대수'].max() + 100  # 예시
real_time_parking['empty_spaces'] = total_spaces - real_time_parking['주차대수']

# 타겟: 1시간 뒤 빈자리 수
real_time_parking['target'] = real_time_parking['empty_spaces'].shift(-1)

# 결측치 제거
real_time_parking = real_time_parking.dropna()

# Model Selection and Training

In [ ]:
# 피처 선택
features = ['hour', 'day_of_week', 'month', 'lag_1h_parking']
X = real_time_parking[features]
y = real_time_parking['target']

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 선택: Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)

# 학습
model.fit(X_train, y_train)

# Model Evaluation and Prediction

In [ ]:
# 예측
y_pred = model.predict(X_test)

# 평가 지표
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'Mean Absolute Error (MAE): {mae:.2f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.2f}')

# 실제 vs 예측 시각화
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Empty Spaces')
plt.ylabel('Predicted Empty Spaces')
plt.title('Actual vs Predicted Empty Spaces')
plt.show()

# 1시간 뒤 예측 예시 (최신 데이터 사용)
latest_features = X.iloc[-1:].copy()
predicted_empty = model.predict(latest_features)[0]
print(f'1시간 뒤 예상 빈자리 수: {predicted_empty:.0f}')